# 06.08 - GAN training stability practice

**Notebook type:** Solution notebook with completed exercises, smoke checks, and test cases.

**Daily output:** GAN update-step practice and stability diagnosis.

This lesson isolates the mechanics that most often break GAN training: detach boundaries, alternating optimizers, gradient flow, and interpreting unstable loss traces.

## Core Ideas

The discriminator learns from real samples and detached generator outputs. Detaching prevents the discriminator update from changing the generator. The generator then learns through a temporarily frozen discriminator. GAN losses are adversarial signals, not ordinary validation metrics: a low discriminator loss may mean the discriminator is overpowering the generator, while nearly constant losses can accompany collapse.

In [ ]:
import numpy as np
import torch
from torch import nn

SEED = 6
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LATENT_DIM = 4

## Prepared Toy Distribution

The real data are noisy points around a unit circle. A two-dimensional fixture keeps every update fast enough to rerun while debugging.

In [ ]:
angles = torch.linspace(0.0, 2.0 * float(np.pi), 65)[:-1]
real_samples = torch.stack([torch.cos(angles), torch.sin(angles)], dim=1)
real_samples += 0.03 * torch.randn_like(real_samples)
fixed_noise = torch.randn(16, LATENT_DIM)
print("real samples:", real_samples.shape, real_samples.dtype, real_samples.device)

## Exercise 06-A: Build the two networks

Use `BCEWithLogitsLoss`, so the discriminator must return raw logits without a final sigmoid.

**Return structure — `build_toy_gan`:** A dictionary with `generator` and `discriminator`, each an `nn.Module` on the requested device. The generator maps `[N,latent_dim]` to `[N,2]`; the discriminator maps `[N,2]` to logits of shape `[N,1]`.

In [ ]:
def build_toy_gan(latent_dim=LATENT_DIM, hidden_dim=16, device=DEVICE):
    generator = nn.Sequential(nn.Linear(latent_dim, hidden_dim), nn.ReLU(), nn.Linear(hidden_dim, 2)).to(device)
    discriminator = nn.Sequential(nn.Linear(2, hidden_dim), nn.LeakyReLU(0.2), nn.Linear(hidden_dim, 1)).to(device)
    return {"generator": generator, "discriminator": discriminator}


# Smoke check: build both modules and verify their boundary shapes.
gan_models = build_toy_gan()
print("G/D shapes:", gan_models["generator"](fixed_noise.to(DEVICE)).shape, gan_models["discriminator"](real_samples[:4].to(DEVICE)).shape)

## Exercise 06-B: Perform one discriminator update

Generate fake samples under `torch.no_grad()` or detach them before calculating discriminator loss.

**Return structure — `discriminator_step`:** A dictionary containing Python floats `d_loss`, `real_loss`, `fake_loss`, and `d_grad_norm`. The function updates only discriminator parameters.

In [ ]:
def discriminator_step(models, optimizer, real_batch, noise):
    generator, discriminator = models["generator"], models["discriminator"]
    loss_function = nn.BCEWithLogitsLoss()
    optimizer.zero_grad()
    with torch.no_grad():
        fake_batch = generator(noise)
    real_logits = discriminator(real_batch)
    fake_logits = discriminator(fake_batch)
    real_loss = loss_function(real_logits, torch.ones_like(real_logits))
    fake_loss = loss_function(fake_logits, torch.zeros_like(fake_logits))
    loss = real_loss + fake_loss
    loss.backward()
    grad_norm = sum(float(parameter.grad.detach().norm()) for parameter in discriminator.parameters() if parameter.grad is not None)
    optimizer.step()
    return {"d_loss": float(loss.detach()), "real_loss": float(real_loss.detach()), "fake_loss": float(fake_loss.detach()), "d_grad_norm": grad_norm}


# Smoke check: run one isolated discriminator update.
d_optimizer = torch.optim.Adam(gan_models["discriminator"].parameters(), lr=0.002)
d_record = discriminator_step(gan_models, d_optimizer, real_samples[:16].to(DEVICE), fixed_noise.to(DEVICE))
print("D record:", d_record)

## Exercise 06-C: Perform one generator update

Freeze discriminator parameters during this step while preserving gradients through its operations to the fake samples.

**Return structure — `generator_step`:** A dictionary containing Python floats `g_loss` and `g_grad_norm`. The function updates only generator parameters and restores discriminator parameters to trainable state.

In [ ]:
def generator_step(models, optimizer, noise):
    generator, discriminator = models["generator"], models["discriminator"]
    loss_function = nn.BCEWithLogitsLoss()
    for parameter in discriminator.parameters():
        parameter.requires_grad_(False)
    optimizer.zero_grad()
    fake_logits = discriminator(generator(noise))
    loss = loss_function(fake_logits, torch.ones_like(fake_logits))
    loss.backward()
    grad_norm = sum(float(parameter.grad.detach().norm()) for parameter in generator.parameters() if parameter.grad is not None)
    optimizer.step()
    for parameter in discriminator.parameters():
        parameter.requires_grad_(True)
    return {"g_loss": float(loss.detach()), "g_grad_norm": grad_norm}


# Smoke check: run one generator update through the discriminator.
g_optimizer = torch.optim.Adam(gan_models["generator"].parameters(), lr=0.002)
g_record = generator_step(gan_models, g_optimizer, fixed_noise.to(DEVICE))
print("G record:", g_record)

## Exercise 06-D: Diagnose a short history

Use transparent rules as debugging prompts, not as universal GAN-quality claims.

**Return structure — `diagnose_gan_history`:** A dictionary with integer `steps`, float `last_d_loss`, float `last_g_loss`, and `warnings` as a `list[str]`.

In [ ]:
def diagnose_gan_history(history):
    if not history:
        raise ValueError("history must contain at least one update")
    warnings = []
    last = history[-1]
    if last["d_loss"] < 0.25:
        warnings.append("discriminator may be overpowering the generator")
    if last["g_loss"] > 3.0:
        warnings.append("generator gradients may be uninformative")
    if last["d_grad_norm"] < 1e-6 or last["g_grad_norm"] < 1e-6:
        warnings.append("vanishing gradient detected")
    if not warnings:
        warnings.append("no threshold warning; inspect generated diversity next")
    return {"steps": len(history), "last_d_loss": float(last["d_loss"]), "last_g_loss": float(last["g_loss"]), "warnings": warnings}


# Smoke check and full prepared-data evidence: collect eight aligned update pairs.
gan_history = []
for step in range(8):
    indices = torch.arange(step * 8, step * 8 + 16) % len(real_samples)
    step_noise = torch.randn(16, LATENT_DIM, device=DEVICE)
    d_values = discriminator_step(gan_models, d_optimizer, real_samples[indices].to(DEVICE), step_noise)
    g_values = generator_step(gan_models, g_optimizer, step_noise)
    gan_history.append({**d_values, **g_values})
stability_report = diagnose_gan_history(gan_history)
print("stability report:", stability_report)

## Test Cases

**Return structure — `run_day06_tests`:** Returns `None`; assertions and `Day 06 tests passed` communicate success.

In [ ]:
def run_day06_tests():
    check_models = build_toy_gan(device=torch.device("cpu"))
    assert set(check_models) == {"generator", "discriminator"}
    assert check_models["generator"](torch.zeros(3, LATENT_DIM)).shape == (3, 2)
    assert check_models["discriminator"](torch.zeros(3, 2)).shape == (3, 1)
    before_g = [parameter.detach().clone() for parameter in check_models["generator"].parameters()]
    before_d = [parameter.detach().clone() for parameter in check_models["discriminator"].parameters()]
    opt_d = torch.optim.SGD(check_models["discriminator"].parameters(), lr=0.01)
    opt_g = torch.optim.SGD(check_models["generator"].parameters(), lr=0.01)
    d_test = discriminator_step(check_models, opt_d, real_samples[:8], torch.randn(8, LATENT_DIM))
    assert any(not torch.equal(a, b) for a, b in zip(before_d, check_models["discriminator"].parameters()))
    assert all(torch.equal(a, b) for a, b in zip(before_g, check_models["generator"].parameters()))
    g_test = generator_step(check_models, opt_g, torch.randn(8, LATENT_DIM))
    assert d_test["d_grad_norm"] > 0 and g_test["g_grad_norm"] > 0
    assert set(stability_report) == {"steps", "last_d_loss", "last_g_loss", "warnings"}
    assert stability_report["steps"] == 8 and stability_report["warnings"]
    print("Day 06 tests passed")


run_day06_tests()

## Day 06 Checklist

- [ ] Keep generator gradients out of the discriminator update.
- [ ] Freeze discriminator parameters during the generator update.
- [ ] Log losses and gradient norms together.
- [ ] Treat warning thresholds as debugging prompts.
- [ ] Run the test cases.